# Limpieza reproducible · cuaderno del proyecto (sesión 5)
**Estud-IA · Análisis de Datos Básico**

Este cuaderno **parte siempre del original** y aplica tus reglas de limpieza una por una. Cada regla anota sola cuántas filas afecta. Al final, los conteos tienen que cuadrar: **original = limpias + quitadas + a revisión**.

Corre las celdas en orden con ▶ (o Shift + Enter). Si Colab avisa que el cuaderno viene de GitHub, elige **«Ejecutar de todos modos»**.

**Tres reglas de oro**
1. El archivo de `datos/originales/` **no se toca**. Todo sale de aquí, con código.
2. Una fila que no sabes cómo arreglar **no se inventa ni se borra en silencio**: va **a revisión**.
3. Si te equivocas o cambias una regla: menú **Entorno de ejecución › Reiniciar y ejecutar todo**. Tiene que dar lo mismo cada vez.

Con el enlace que viene (la papelería sucia, 11 filas) debe quedar: **8 limpias, 1 quitada y 2 a revisión**.

In [38]:
# Celda 1 · Preparar (no cambies nada)
try:
    import duckdb
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb
import re
import pandas as pd

def cargar(ruta, hoja=0):
    """Lee un CSV (coma o punto y coma) o un Excel. Acepta el enlace Raw o el de la página de GitHub."""
    ruta = re.sub(r"^https://github\.com/([^/]+)/([^/]+)/blob/", r"https://raw.githubusercontent.com/\1/\2/", ruta.strip())
    if ruta.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(ruta, sheet_name=hoja)
    for codificacion in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(ruta, sep=None, engine="python", encoding=codificacion)
        except UnicodeDecodeError:
            continue
        except pd.errors.ParserError:
            raise ValueError("No pude leerlo como tabla. Revisa que el enlace sea el de tu CSV (botón Raw).") from None
    raise ValueError("No pude leer el archivo: revisa que sea un CSV")

def sql(consulta):
    """Corre una consulta SQL sobre las tablas cargadas (original, base, analisis) y muestra el resultado."""
    resultado = duckdb.sql(consulta).df()
    for col in resultado.select_dtypes("float").columns:
        if resultado[col].dropna().mod(1).eq(0).all():
            resultado[col] = resultado[col].astype("Int64")   # 45000 en lugar de 45000.0
    return resultado

def _filas(n):
    return f"{n} fila" if n == 1 else f"{n} filas"

def perfil(tabla):
    """Cuántas filas y columnas hay, cuántas filas repetidas y cuántos vacíos tiene cada columna."""
    print(f"{_filas(len(tabla))} · {len(tabla.columns)} columnas · filas idénticas a otra: {int(tabla.duplicated().sum())}")
    return pd.DataFrame({
        "tipo": [str(t) for t in tabla.dtypes],
        "vacíos": tabla.isna().sum(),
        "valores distintos": tabla.nunique(),
        "ejemplo": [tabla[c].dropna().iloc[0] if tabla[c].notna().any() else "" for c in tabla.columns],
    })

def empezar(tabla):
    """Copia el original y deja la bitácora en cero. Todo vuelve a empezar desde aquí."""
    global base, quitadas, revision, bitacora, _original
    _original = tabla.copy()
    base = tabla.copy()
    quitadas = tabla.iloc[0:0].copy()
    revision = tabla.iloc[0:0].copy()
    bitacora = []
    print(f"Empezamos desde el original: {len(base)} filas.")

def _ya_esta(nombre):
    if any(b["regla"] == nombre for b in bitacora):
        raise RuntimeError(f"La regla «{nombre}» ya se aplicó. Corre otra vez desde la celda 4 (Empezar) para no aplicarla dos veces.")

def regla(nombre, filas, motivo, accion="quitar"):
    """Aplica una regla a las filas marcadas con True.
    accion = "quitar": salen de la base (por ejemplo, copias exactas).
    accion = "revisar": salen de la base y van al archivo de revisión (no se inventan ni se pierden)."""
    global base, quitadas, revision
    _ya_esta(nombre)
    filas = pd.Series(filas, index=base.index).fillna(False).astype(bool)
    n, antes = int(filas.sum()), len(base)
    afectadas = _original.loc[base.index[filas]].copy()   # con los valores originales, para poder revisarlas
    afectadas["regla"] = nombre
    if accion == "quitar":
        quitadas = pd.concat([quitadas, afectadas])
    elif accion == "revisar":
        revision = pd.concat([revision, afectadas])
    else:
        raise ValueError('accion debe ser "quitar" o "revisar"')
    base = base[~filas].copy()
    bitacora.append({"regla": nombre, "filas afectadas": n, "acción": accion, "motivo": motivo, "antes": antes, "después": len(base)})
    print(f"{nombre}: {_filas(n)} ({accion}). Antes {antes}, ahora {len(base)}.")
    return afectadas.head(10)

def unificar(columna, motivo="El mismo valor estaba escrito de varias formas"):
    """Quita espacios sobrantes y une las formas que solo cambian en mayúsculas: deja la forma más común."""
    nombre = f"Unificar escritura de {columna}"
    _ya_esta(nombre)
    antes = base[columna].copy()
    texto = antes.astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
    clave = texto.str.lower()
    mas_comun = texto.groupby(clave).agg(lambda s: s.value_counts().index[0])
    nuevo = clave.map(mas_comun)
    nuevo = nuevo.where(antes.notna(), antes)
    cambiadas = (antes.astype("string") != nuevo.astype("string")).fillna(False)
    base[columna] = nuevo
    bitacora.append({"regla": nombre, "filas afectadas": int(cambiadas.sum()), "acción": "corregir", "motivo": motivo, "antes": len(base), "después": len(base)})
    print(f"{nombre}: filas corregidas: {int(cambiadas.sum())}. Grupos antes: {antes.nunique()}, después: {base[columna].nunique()}.")
    return pd.DataFrame({"antes": antes[cambiadas], "después": nuevo[cambiadas]}).drop_duplicates().head(10)

def reemplazar(columna, cambios, motivo="El mismo valor estaba escrito de otra forma"):
    """Cambia valores exactos por otros: reemplazar("municipio", {"Estrella": "La Estrella"}).
    Sirve cuando unificar() no alcanza: tildes, abreviaturas o nombres distintos de la misma cosa."""
    nombre = f"Reemplazar en {columna}: " + ", ".join(f"{a} -> {b}" for a, b in cambios.items())
    _ya_esta(nombre)
    antes = base[columna].copy()
    base[columna] = base[columna].replace(cambios)
    cambiadas = (antes.astype("string") != base[columna].astype("string")).fillna(False)
    bitacora.append({"regla": nombre, "filas afectadas": int(cambiadas.sum()), "acción": "corregir", "motivo": motivo, "antes": len(base), "después": len(base)})
    print(f"{nombre}: filas corregidas: {int(cambiadas.sum())}. Valores distintos antes: {antes.nunique()}, después: {base[columna].nunique()}.")
    return pd.DataFrame({"antes": antes[cambiadas], "después": base[columna][cambiadas]}).value_counts().rename("filas").reset_index()

def a_numero(columna, miles=None):
    """Convierte una columna a número. Si los miles vienen con punto o coma, escribe miles="." o miles=",".
    Devuelve True en las filas que NO se pudieron convertir (y no estaban vacías): decide qué hacer con ellas."""
    texto = base[columna].astype("string").str.strip()
    if miles:
        texto = texto.str.replace(miles, "", regex=False)
    numero = pd.to_numeric(texto, errors="coerce")
    malas = numero.isna() & base[columna].notna()
    if numero.dropna().mod(1).eq(0).all():
        numero = numero.astype("Int64")   # enteros sin decimales
    base[columna] = numero
    print(f"{columna} ahora es número. Filas que no se pudieron convertir: {int(malas.sum())}.")
    return malas

def a_fecha(columna, formato):
    """Convierte una columna a fecha con un formato explícito (por ejemplo "%Y-%m-%d").
    Devuelve True en las filas que NO se pudieron convertir (y no estaban vacías)."""
    fecha = pd.to_datetime(base[columna].astype("string").str.strip(), format=formato, errors="coerce")
    malas = fecha.isna() & base[columna].notna()
    base[columna] = fecha
    print(f"{columna} ahora es fecha. Filas que no se pudieron leer con el formato {formato}: {int(malas.sum())}.")
    return malas

def ver_bitacora():
    """La bitácora de limpieza como tabla y como líneas para pegar en bitacora.md."""
    tabla = pd.DataFrame(bitacora)
    for b in bitacora:
        print(f"- {b['regla']} · {_filas(b['filas afectadas'])} · {b['acción']} · {b['motivo']}")
    return tabla

def conciliar(original):
    """Comprueba que no se perdió ninguna fila: original = limpias + quitadas + a revisión."""
    total = len(base) + len(quitadas) + len(revision)
    print(f"Original {len(original)} = limpias {len(base)} + quitadas {len(quitadas)} + a revisión {len(revision)}  →  {total}")
    print("Cuadra." if total == len(original) else "NO CUADRA: alguna fila se perdió o se duplicó. Corre todo desde la celda 4.")

def guardar(nombre):
    """Guarda la base limpia y el archivo de revisión como CSV y los descarga a tu computador."""
    base.to_csv(f"{nombre}_limpio.csv", index=False)
    revision.to_csv(f"{nombre}_revision.csv", index=False)
    print(f"Guardé {nombre}_limpio.csv ({len(base)} filas) y {nombre}_revision.csv ({len(revision)} filas).")
    try:
        from google.colab import files
        files.download(f"{nombre}_limpio.csv")
        files.download(f"{nombre}_revision.csv")
    except Exception:
        print("Fuera de Colab: los archivos quedaron en la carpeta del cuaderno.")

print("Listo. Sigue con la celda 2.")

Listo. Sigue con la celda 2.


In [39]:
import urllib.request
import csv
import pandas as pd

# 1. Descargamos el archivo original de tu GitHub
url_original = "https://raw.githubusercontent.com/dianagironc1/proyecto-e10/main/datos/originales/sales_data_sample.csv"
ruta_local_sucia = "sales_data_sucia.csv"
ruta_local_limpia = "sales_data_limpio_estructura.csv"

urllib.request.urlretrieve(url_original, ruta_local_sucia)

# 2. Reconstruimos el CSV reparando comillas problemáticas línea por línea
filas_corregidas = []
with open(ruta_local_sucia, "r", encoding="utf-8-sig", errors="ignore") as f_in:
    # Usamos el analizador estándar de CSV para evitar romper celdas que contienen comas internas
    lector = csv.reader(f_in, delimiter=',', skipinitialspace=True)
    for fila in lector:
        # Limpiamos espacios extras alrededor de cada campo
        fila_limpia = [campo.strip() for campo in fila]
        filas_corregidas.append(fila_limpia)

# 3. Guardamos el archivo limpio estructurado de manera que pandas lo interprete sin problemas
with open(ruta_local_limpia, "w", encoding="utf-8", newline="") as f_out:
    escritor = csv.writer(f_out, delimiter=',', quoting=csv.QUOTE_MINIMAL)
    escritor.writerows(filas_corregidas)

print("¡Estructura de columnas corregida perfectamente! Generado: 'sales_data_limpio_estructura.csv'")

¡Estructura de columnas corregida perfectamente! Generado: 'sales_data_limpio_estructura.csv'


In [28]:
import csv
import pandas as pd

ruta_local_sucia = "sales_data_sucia.csv"
ruta_local_limpia = "sales_data_limpio_estructura.csv"

filas_corregidas = []

with open(ruta_local_sucia, "r", encoding="utf-8-sig", errors="ignore") as f_in:
    lector = csv.reader(f_in, delimiter=',', skipinitialspace=True)
    cabecera = next(lector)

    # Identificamos los índices de las columnas a eliminar y la columna a limpiar
    idx_address1 = cabecera.index('ADDRESSLINE1') if 'ADDRESSLINE1' in cabecera else None
    idx_address2 = cabecera.index('ADDRESSLINE2') if 'ADDRESSLINE2' in cabecera else None
    idx_customer = cabecera.index('CUSTOMERNAME') if 'CUSTOMERNAME' in cabecera else None

    # Construimos la nueva cabecera sin ADDRESSLINE1 ni ADDRESSLINE2
    nueva_cabecera = [col for idx, col in enumerate(cabecera) if idx not in (idx_address1, idx_address2)]
    filas_corregidas.append(nueva_cabecera)

    for fila in lector:
        # Limpiamos espacios alrededor de cada campo
        fila_limpia = [campo.strip() for campo in fila]

        # Eliminamos las comas internas en CUSTOMERNAME si la columna existe
        if idx_customer is not None and idx_customer < len(fila_limpia):
            fila_limpia[idx_customer] = fila_limpia[idx_customer].replace(',', '')

        # Filtramos la fila para no incluir las columnas de dirección
        nueva_fila = [campo for idx, campo in enumerate(fila_limpia) if idx not in (idx_address1, idx_address2)]
        filas_corregidas.append(nueva_fila)

# Guardamos el archivo resultante
with open(ruta_local_limpia, "w", encoding="utf-8", newline="") as f_out:
    escritor = csv.writer(f_out, delimiter=',', quoting=csv.QUOTE_MINIMAL)
    escritor.writerows(filas_corregidas)

# Recargamos la variable 'original' para que se actualice en todo el cuaderno
original = pd.read_csv(ruta_local_limpia, sep=",", encoding="utf-8")
print(f"¡Archivo limpio generado! Ahora tiene {len(original)} filas y {len(original.columns)} columnas.")
print("Columnas restantes:", list(original.columns))
print("\nEjemplo de nombres de cliente limpios de comas:")
display(original['CUSTOMERNAME'].unique()[:5])

¡Archivo limpio generado! Ahora tiene 2823 filas y 23 columnas.
Columnas restantes: ['ORDERNUMBER', 'QUANTITYORDERED', 'PRICEEACH', 'ORDERLINENUMBER', 'SALES', 'ORDERDATE', 'STATUS', 'QTR_ID', 'MONTH_ID', 'YEAR_ID', 'PRODUCTLINE', 'MSRP', 'PRODUCTCODE', 'CUSTOMERNAME', 'PHONE', 'CITY', 'STATE', 'POSTALCODE', 'COUNTRY', 'TERRITORY', 'CONTACTLASTNAME', 'CONTACTFIRSTNAME', 'DEALSIZE']

Ejemplo de nombres de cliente limpios de comas:


array(['Land of Toys Inc.', 'Reims Collectables', 'Lyon Souveniers',
       'Toys4GrownUps.com', 'Corporate Gift Ideas Co.'], dtype=object)

In [47]:
import csv
import pandas as pd

ruta_local_sucia = "sales_data_sucia.csv"
ruta_local_limpia = "sales_data_limpio_estructura.csv"

filas_corregidas = []

with open(ruta_local_sucia, "r", encoding="utf-8-sig", errors="ignore") as f_in:
    lineas = f_in.readlines()

cabecera_procesada = False
idx_address1, idx_address2, idx_customer = None, None, None
nueva_cabecera = []

for i, linea_cruda in enumerate(lineas):
    linea_ajustada = linea_cruda.strip()

    # 1. Limpieza clave preventiva: eliminamos la comilla inicial huérfana en ORDERNUMBER si existe
    if linea_ajustada.startswith('"'):
        linea_ajustada = linea_ajustada[1:]

    # Usamos csv.reader en la línea limpia para extraer correctamente cada columna
    lector = csv.reader([linea_ajustada], delimiter=',', skipinitialspace=True)
    fila = next(lector)

    if not cabecera_procesada:
        cabecera = [c.strip() for c in fila]
        idx_address1 = cabecera.index('ADDRESSLINE1') if 'ADDRESSLINE1' in cabecera else None
        idx_address2 = cabecera.index('ADDRESSLINE2') if 'ADDRESSLINE2' in cabecera else None
        idx_customer = cabecera.index('CUSTOMERNAME') if 'CUSTOMERNAME' in cabecera else None

        nueva_cabecera = [col for idx, col in enumerate(cabecera) if idx not in (idx_address1, idx_address2)]
        filas_corregidas.append(nueva_cabecera)
        cabecera_procesada = True
        continue

    # Si la fila tiene menos columnas de las debidas por un salto de línea corrupto o comillas internas cruzadas
    # removemos las comillas problemáticas para restaurar el formato plano original
    if len(fila) < len(cabecera):
        linea_plana = linea_ajustada.replace('"', '')
        lector = csv.reader([linea_plana], delimiter=',')
        fila = next(lector)

    fila_limpia = []
    for idx, campo in enumerate(fila):
        if idx in (idx_address1, idx_address2):
            continue

        campo_limpio = campo.strip().replace('"', '').replace("'", "")

        # Quitamos comas en nombres de clientes u otros campos descriptivos
        if idx == idx_customer or not campo_limpio.replace('.', '', 1).isdigit():
            campo_limpio = campo_limpio.replace(',', '')

        fila_limpia.append(campo_limpio)

    # Ajustamos cualquier desfase de tamaño rellenando con nulos si hiciese falta
    while len(fila_limpia) < len(nueva_cabecera):
        fila_limpia.append("")

    filas_corregidas.append(fila_limpia[:len(nueva_cabecera)])

# Guardamos el archivo completamente estructurado y limpio
with open(ruta_local_limpia, "w", encoding="utf-8", newline="") as f_out:
    escritor = csv.writer(f_out, delimiter=',')
    escritor.writerows(filas_corregidas)

# Recargamos en original
original = pd.read_csv(ruta_local_limpia, sep=",", encoding="utf-8")
print(f"¡Limpieza estructural perfecta completada! Archivo guardado con {len(original)} filas y {len(original.columns)} columnas.")

¡Limpieza estructural perfecta completada! Archivo guardado con 2823 filas y 23 columnas.


## 1. Carga el ORIGINAL

Pega el enlace de tu archivo de **`datos/originales/`** (la página de GitHub o el botón **Raw**). Si es un Excel con varias hojas, escribe el nombre de la hoja en `HOJA`.

Con la papelería sucia: **11 filas y 6 columnas**.

In [48]:
# Celda 2 · Cargar el original (no se modifica)
ENLACE = "https://github.com/dianagironc1/proyecto-e10/blob/main/datos/originales/sales_data_sample.csv"
HOJA = 0   # solo para Excel con varias hojas: el nombre entre comillas

original = cargar(ENLACE, HOJA)
print(f"Original: {len(original)} filas y {len(original.columns)} columnas.")
original.head()

Original: 2823 filas y 25 columnas.


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30.0,95.70,2.0,2871.00,2/24/2003 0:00,Shipped,1.0,2.0,2003.0,...,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34.0,81.35,5.0,2765.90,5/7/2003 0:00,Shipped,2.0,5.0,2003.0,...,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41.0,94.74,2.0,3884.34,7/1/2003 0:00,Shipped,3.0,7.0,2003.0,...,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45.0,83.26,6.0,3746.70,8/25/2003 0:00,Shipped,3.0,8.0,2003.0,...,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,10159,49.0,100.00,14.0,5205.27,10/10/2003 0:00,Shipped,4.0,10.0,2003.0,...,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium


In [49]:
# Celda 2 (Modificada para usar el archivo corregido local)
ENLACE = "sales_data_limpio_estructura.csv"
HOJA = 0

import pandas as pd
# Cargamos el archivo limpio asegurando la interpretación estándar del estándar CSV
original = pd.read_csv(ENLACE, sep=",", encoding="utf-8")
print(f"Original Corregido: {len(original)} filas y {len(original.columns)} columnas.")
original.head()

Original Corregido: 2823 filas y 23 columnas.


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,CUSTOMERNAME,PHONE,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2/24/2003 0:00,Shipped,1,2,2003,...,Land of Toys Inc.,2125557818,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,5/7/2003 0:00,Shipped,2,5,2003,...,Reims Collectables,26.47.1555,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,7/1/2003 0:00,Shipped,3,7,2003,...,Lyon Souveniers,+33 1 46 62 7555,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,8/25/2003 0:00,Shipped,3,8,2003,...,Toys4GrownUps.com,6265557265,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,10/10/2003 0:00,Shipped,4,10,2003,...,Corporate Gift Ideas Co.,6505551386,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium


## 2. Mira antes de limpiar

El perfil te dice qué revisar: filas idénticas, columnas con vacíos y columnas que deberían ser número o fecha pero salen como texto (`object` o `str`).

Con la papelería sucia: **1 fila idéntica a otra**, `cantidad` y `fecha` salen como texto.

In [50]:
# Celda 3 · Perfil del original corregido
import pandas as pd

# Nos aseguramos de leer la última versión limpia guardada localmente
original = pd.read_csv('sales_data_limpio_estructura.csv', sep=",", encoding="utf-8")
perfil(original)

2823 filas · 23 columnas · filas idénticas a otra: 0


,tipo,vacíos,valores distintos,ejemplo
ORDERNUMBER,int64,0,307,10107
QUANTITYORDERED,int64,0,58,30
PRICEEACH,float64,0,1016,95.7
ORDERLINENUMBER,int64,0,18,2
SALES,float64,0,2763,2871.0
ORDERDATE,object,0,252,2/24/2003 0:00
STATUS,object,0,6,Shipped
QTR_ID,int64,0,4,1
MONTH_ID,int64,0,12,2
YEAR_ID,int64,0,3,2003


In [44]:
import pandas as pd

# Cargamos el archivo que acabamos de limpiar de manera masiva
df_limpio = pd.read_csv('sales_data_limpio_estructura.csv', sep=",", encoding="utf-8")

# Verificamos si en la columna ORDERNUMBER queda algún elemento con comillas y más de 5 caracteres
ordernumber_texto = df_limpio['ORDERNUMBER'].astype(str)
con_comillas_y_largos_limpio = ordernumber_texto.apply(lambda x: (('"' in x) or ("'" in x)) and len(x) > 5)

cantidad_encontrada = con_comillas_y_largos_limpio.sum()
ejemplos_limpios = df_limpio[con_comillas_y_largos_limpio]['ORDERNUMBER'].head(5).tolist()

print(f"Resultados en el archivo limpio y reestructurado:")
print(f"- Cantidad de ORDERNUMBER entre comillas y > 5 caracteres: {cantidad_encontrada}")
if ejemplos_limpios:
    print(f"- Ejemplos encontrados: {ejemplos_limpios}")
else:
    print("- ¡Excelente! No se encontraron valores mal formateados o con comas/comillas cruzadas en ORDERNUMBER.")

# También mostramos una muestra rápida de los valores actuales para dar tranquilidad
print("\nMuestra de los primeros 5 ORDERNUMBER en la tabla final:")
display(df_limpio['ORDERNUMBER'].head(5))

Resultados en el archivo limpio y reestructurado:
- Cantidad de ORDERNUMBER entre comillas y > 5 caracteres: 1315
- Ejemplos encontrados: ['101802986.1392497.7711/11/2003 0:00Shipped4112003Motorcycles95S10_1678Daedalus Designs Imports20.16.1555"184 chausse de Tournai"Lille59000FranceEMEARanceMartineSmall', '101884810015512.3211/18/2003 0:00Shipped4112003Motorcycles95S10_1678Herkku Gifts+47 2267 3215"Drammen 121 PR 744 Sentrum"BergenN 5804NorwayEMEAOeztanVeyselMedium', '1021141100144708.441/15/2004 0:00Shipped112004Motorcycles95S10_1678Auto Canal Petit(1) 47.55.6555"25 rue Lauriston"Paris75016FranceEMEAPerrierDominiqueMedium', '102233710013965.662/20/2004 0:00Shipped122004Motorcycles95S10_1678"Australian Collectors Co."03 9520 4555636 St Kilda RoadLevel 3MelbourneVictoria3004AustraliaAPACFergusonPeterMedium', '102754592.8314177.357/23/2004 0:00Shipped372004Motorcycles95S10_1678La Rochelle Gifts40.67.8555"67 rue des Cinquante Otages"Nantes44000FranceEMEALabruneJanineMedium']

Muestra de 

,ORDERNUMBER
0,10107
1,10121
2,10134
3,10145
4,10159


In [51]:
# Análisis detallado de problemas estructurales
print("=== 1. Valores únicos en columnas categóricas ===")
for col in ['STATUS', 'DEALSIZE', 'COUNTRY']:
    print(f"\nValores en {col}:")
    display(original[col].value_counts(dropna=False).head(10))

print("\n=== 2. Ejemplo de registros con valores vacíos en campos numéricos clave (SALES o PRICEEACH) ===")
vacios_criticos = original[original['SALES'].isna() | original['PRICEEACH'].isna()]
print(f"Filas con ventas o precios vacíos: {len(vacios_criticos)}")
display(vacios_criticos[['ORDERNUMBER', 'QUANTITYORDERED', 'PRICEEACH', 'SALES']].head(5))

=== 1. Valores únicos en columnas categóricas ===

Valores en STATUS:


,count
STATUS,
Shipped,2617
Cancelled,60
Resolved,47
On Hold,44
In Process,41
Disputed,14



Valores en DEALSIZE:


,count
DEALSIZE,
Medium,745
Small,673
Diego,259
Large,90
Michael,59
Peter,55
Elizabeth,55
Janine,53
Victoria,51



Valores en COUNTRY:


,count
COUNTRY,
USA,969
28034,259
NaN,192
Italy,113
France,81
Denmark,63
44000,60
3004,55
Austria,55



=== 2. Ejemplo de registros con valores vacíos en campos numéricos clave (SALES o PRICEEACH) ===
Filas con ventas o precios vacíos: 0


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,SALES


In [52]:
# Inspeccionemos las filas que tienen todos estos vacíos para ver si realmente están vacías
filas_vacias = original[original['SALES'].isna()]
print(f"Cantidad de filas con ventas vacías: {len(filas_vacias)}")
print("\nEjemplo de las primeras 5 filas con campos vacíos:")
display(filas_vacias.head(5))

print("\nValores únicos en 'ORDERNUMBER' para estas filas vacías:")
display(filas_vacias['ORDERNUMBER'].value_counts().head(10))

Cantidad de filas con ventas vacías: 0

Ejemplo de las primeras 5 filas con campos vacíos:


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,CUSTOMERNAME,PHONE,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE



Valores únicos en 'ORDERNUMBER' para estas filas vacías:


,count
ORDERNUMBER,


In [26]:
import csv

con_comillas_y_largos = 0
ejemplos = []

with open('sales_data_sucia.csv', 'r', encoding='utf-8-sig', errors='ignore') as f:
    # Leemos línea por línea para analizar el texto crudo tal como viene en el CSV
    for i, linea in enumerate(f):
        if i == 0:
            continue # Saltamos la cabecera

        # Obtenemos el primer elemento que corresponde a ORDERNUMBER en el texto crudo
        primer_campo = linea.split(',')[0].strip()

        # Verificamos si tiene comillas y si su longitud (incluyendo o excluyendo comillas) supera los 5 caracteres
        tiene_comillas = ('"' in primer_campo) or ("'" in primer_campo)

        if tiene_comillas and len(primer_campo) > 5:
            con_comillas_y_largos += 1
            if len(ejemplos) < 5:
                ejemplos.append(primer_campo)

print(f"Resultados del análisis crudo:")
print(f"- Cantidad de ORDERNUMBER entre comillas con más de 5 caracteres: {con_comillas_y_largos}")
if ejemplos:
    print(f"- Ejemplos encontrados: {ejemplos}")

Resultados del análisis crudo:
- Cantidad de ORDERNUMBER entre comillas con más de 5 caracteres: 1315
- Ejemplos encontrados: ['"10180', '"10188', '"10211', '"10223', '"10275']


## 3. Empieza (siempre desde el original)

Cada vez que cambies una regla, vuelve a correr **desde aquí** hacia abajo. Si corres una regla dos veces, el cuaderno te avisa.

In [59]:
# Celda 4 · Empezar desde el original
empezar(original)

Empezamos desde el original: 2823 filas.


## 4. Tus reglas, una por celda

Cada regla tiene **nombre**, las **filas** que afecta (True/False), un **motivo** y una **acción**:
- `"quitar"`: salen de la base. Solo para lo que estás seguro de que sobra (una copia exacta).
- `"revisar"`: salen de la base y van al archivo de revisión. Para lo que no sabes cómo arreglar sin inventar (una fecha imposible).

Las reglas de abajo son las de la papelería. **Cámbialas por las tuyas**: usa lo que anotaste en la bitácora el martes.

In [54]:
# Celda 5 · Regla 1: quitar copias exactas
# Una copia exacta es una fila igualita a otra en TODAS las columnas.
regla("Copias exactas",
      base.duplicated(),
      motivo="La misma venta registrada dos veces",
      accion="quitar")

Copias exactas: 0 filas (quitar). Antes 2823, ahora 2823.


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,PHONE,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE,regla


**Antes de quitar copias, pregúntate:** ¿es el mismo registro dos veces o son dos cosas distintas que se ven iguales (dos ventas iguales el mismo día)? Si tu base **no tiene un código** que identifique cada fila, quizá no debas quitarlas: anótalo como límite.

In [60]:
# Celda 6 · Regla 2: unificar cómo está escrita una columna de texto
# Cambiado "estado" por "STATUS" que es la columna real de categorías en tu set de datos.
unificar("STATUS")

Unificar escritura de STATUS: filas corregidas: 0. Grupos antes: 6, después: 6.


,antes,después


### Fechas y números: primero se convierte, después se decide

`a_fecha` y `a_numero` convierten la columna y te devuelven las filas que **no** se pudieron convertir. Esas filas van a revisión con una regla.

Formatos de fecha comunes (escribe el que tiene tu archivo):

| Así se ve en tu archivo | Formato |
|---|---|
| 2026-08-01 | `"%Y-%m-%d"` |
| 01/08/2026 (día/mes/año) | `"%d/%m/%Y"` |
| 08/01/2026 (mes/día/año) | `"%m/%d/%Y"` |
| 03/22/2019 12:00:00 AM | `"%m/%d/%Y %I:%M:%S %p"` |
| 2/24/2003 0:00 | `"%m/%d/%Y %H:%M"` |
| 2025-01-01T00:00:00.000 | `"%Y-%m-%dT%H:%M:%S.%f"` |

Si **todas** las fechas salen malas, el formato no es ese: míralo otra vez en la celda 2.

In [61]:
# Celda 7 · Regla 3: fechas que no son fechas
# Cambiado "fecha" por "ORDERDATE" con su formato correspondiente "%m/%d/%Y %H:%M"
malas = a_fecha("ORDERDATE", "%m/%d/%Y %H:%M")
regla("Fecha imposible",
      malas,
      motivo="La fecha no existe; no se inventa",
      accion="revisar")

ORDERDATE ahora es fecha. Filas que no se pudieron leer con el formato %m/%d/%Y %H:%M: 0.
Fecha imposible: 0 filas (revisar). Antes 2823, ahora 2823.


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,PHONE,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE,regla


In [68]:
# Celda 8 · Regla 4: números que no son números
# Cambiado "cantidad" por tu columna "QUANTITYORDERED".
malas = a_numero("QUANTITYORDERED")
regla("Cantidad no numérica",
      malas,
      motivo="La cantidad está en letras o tiene un formato no válido",
      accion="revisar")

QUANTITYORDERED ahora es número. Filas que no se pudieron convertir: 0.


RuntimeError: La regla «Cantidad no numérica» ya se aplicó. Corre otra vez desde la celda 4 (Empezar) para no aplicarla dos veces.

### Regla Opcional: Unificar columnas geográficas (CITY, STATE, COUNTRY)
Esto limpiará espacios en blanco adicionales y unificará variaciones de mayúsculas/minúsculas en los nombres de las ciudades, estados y países.

In [69]:
# Celda 8b · Unificar consistencia geográfica
unificar("CITY")
unificar("STATE")
unificar("COUNTRY")

RuntimeError: La regla «Unificar escritura de CITY» ya se aplicó. Corre otra vez desde la celda 4 (Empezar) para no aplicarla dos veces.

## 5. ¿Cuadra?

**Original = limpias + quitadas + a revisión.** Si no cuadra, alguna regla perdió o duplicó filas: vuelve a correr desde la celda 4.

Con la papelería sucia: **11 = 8 + 1 + 2** (cuadra)

In [64]:
# Celda 9 · Conciliar los conteos
conciliar(original)

Original 2823 = limpias 2823 + quitadas 0 + a revisión 0  →  2823
Cuadra.


In [65]:
# Celda 10 · La bitácora de limpieza (copia las líneas a bitacora.md, sección Limpieza)
ver_bitacora()

- Unificar escritura de STATUS · 0 filas · corregir · El mismo valor estaba escrito de varias formas
- Fecha imposible · 0 filas · revisar · La fecha no existe; no se inventa
- Cantidad no numérica · 0 filas · revisar · La cantidad está en letras o tiene un formato no válido
- Unificar escritura de CITY · 0 filas · corregir · El mismo valor estaba escrito de varias formas
- Unificar escritura de STATE · 0 filas · corregir · El mismo valor estaba escrito de varias formas
- Unificar escritura de COUNTRY · 0 filas · corregir · El mismo valor estaba escrito de varias formas


,regla,filas afectadas,acción,motivo,antes,después
0,Unificar escritura de STATUS,0,corregir,El mismo valor estaba escrito de varias formas,2823,2823
1,Fecha imposible,0,revisar,La fecha no existe; no se inventa,2823,2823
2,Cantidad no numérica,0,revisar,La cantidad está en letras o tiene un formato ...,2823,2823
3,Unificar escritura de CITY,0,corregir,El mismo valor estaba escrito de varias formas,2823,2823
4,Unificar escritura de STATE,0,corregir,El mismo valor estaba escrito de varias formas,2823,2823
5,Unificar escritura de COUNTRY,0,corregir,El mismo valor estaba escrito de varias formas,2823,2823


In [66]:
# Celda 11 · Lo que quedó a revisión (no se pierde: queda en su propio archivo)
revision

,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,PHONE,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE,regla


## 6. Base limpia frente a vista de análisis

La **base limpia** tiene todas las filas válidas. La **vista de análisis** es el pedazo que usa tu pregunta (por ejemplo, solo las ventas completadas). No se borra lo demás: se filtra.

Con la papelería: base limpia **8 filas**; vista de análisis (completadas) **7 filas**, que suman **121.000** pesos (agosto 45.000 y septiembre 76.000).

In [71]:
# Celda 12 · Vista de análisis: cambia la condición por la de tu pregunta
# Corregido: usando la columna 'STATUS' y el valor 'Shipped' (Enviado/Completado)
analisis = base[base["STATUS"] == "Shipped"].copy()
print(f"Base limpia: {len(base)} filas · vista de análisis: {len(analisis)} filas")

Base limpia: 2823 filas · vista de análisis: 2617 filas


## 7. Vuelve a correr tus consultas sobre la base limpia

Las consultas de la sesión 3 ahora van sobre `analisis` (o `base`) en lugar de `datos`. ¿Qué cifra cambió frente a la que tenías? Anótalo en la bitácora: **antes · después · por qué**.

## 8. Guarda y sube por PR

1. Corre la celda 14: descarga `…_limpio.csv` y `…_revision.csv`.
2. Menú **Archivo › Descargar › Descargar .ipynb** (este cuaderno, con tus reglas).
3. En GitHub, en una **rama nueva**: los dos CSV a `datos/limpios/` y el cuaderno a `notebooks/02_limpieza.ipynb` (**Add file › Upload files**).
4. Las líneas de la celda 10 a `bitacora.md` (sección **Limpieza**) y, si usaste IA, a **Uso de IA**.
5. Abre el PR: en la descripción, la conciliación (**original = limpias + quitadas + revisión**) y pide revisión a **santiagom26cc**.

**Prueba final:** menú **Entorno de ejecución › Reiniciar y ejecutar todo**. ¿Te da los mismos números? Entonces es reproducible.

In [75]:
# ¿Cuántas ventas fueron entregadas "Shipped"?
# Datos: datos/originales/sales_data_sample.csv
# Cifra de control: 2823 en total, 1399 completadas.

sql("""
SELECT COUNT(*) AS filas_filtradas
FROM original
WHERE STATUS = 'Shipped'
""")

,filas_filtradas
0,2617


In [78]:
# Celda 14 · Guardar (cambia "ventas" por un nombre corto de tu base)
guardar("sales_data_sample")

Guardé sales_data_sample_limpio.csv (2823 filas) y sales_data_sample_revision.csv (0 filas).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [81]:
# Descargar los archivos limpios generados a tu equipo local
try:
    from google.colab import files
    files.download('sales_data_sample_limpio.csv')
    files.download('sales_data_sample_revision.csv')
except Exception as e:
    print("No se pudo realizar la descarga automática:", e)
    print("Puedes descargarlos manualmente desde el panel de archivos de la izquierda en Colab (icono de carpeta).")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
¿Algo falló? Regla de 3 y, si sigues bloqueado, un ticket en la **Cola de ayuda** con el mensaje de error copiado. Nunca pegues contraseñas ni datos personales.